# Fraud Detection - Data Preprocessing

This notebook implements the data preprocessing pipeline based on the insights gathered during the Exploratory Data Analysis (EDA). The goal is to clean the data, handle missing values, engineer relevant features, and prepare the dataset for modeling.


## 1. Import Libraries

We use standard data science libraries for processing and scikit-learn for transformations.


In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib

## 2. Load and Merge Data

As identified in the EDA, the dataset is split into transaction and identity files. We merge them on `TransactionID` to create a unified dataset.


In [2]:
train_trans = pd.read_csv("../data/raw/train_transaction.csv")
train_id = pd.read_csv("../data/raw/train_identity.csv")

df = train_trans.merge(train_id, on="TransactionID", how="left")

print("Merged Shape:", df.shape)

Merged Shape: (590540, 434)


## 3. Drop Highly Missing Features

During EDA, we observed that many features have a very high percentage of missing values (some over 90%). These columns are unlikely to provide significant predictive power and might introduce noise. We drop features with more than 90% missing values.


In [3]:
missing_percent = df.isnull().mean() * 100

cols_to_drop = missing_percent[missing_percent > 90].index
df.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns with >90% missing values.")
print("After dropping high-missing columns:", df.shape)

Dropped 12 columns with >90% missing values.
After dropping high-missing columns: (590540, 422)


## 4. Feature Engineering

Based on EDA insights, we perform two key transformations on the `TransactionAmt` feature:

1. **Log Transformation**: To handle the high skewness of transaction amounts.
2. **Decimal Extraction**: To capture potential patterns in how transaction values are rounded or entered.


### Log Transform (TransactionAmt)


In [4]:
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

### Decimal Feature


In [5]:
df['TransactionAmt_decimal'] = (
    (df['TransactionAmt'] - df['TransactionAmt'].astype(int)) * 1000
)


## 5. Separate Features and Target

We separate the target variable `isFraud` and drop non-predictive identifiers like `TransactionID`.


In [6]:
target = "isFraud"

X = df.drop(columns=[target, "TransactionID"])
y = df[target]


## 6. Handle Missing Values

We apply basic imputation strategies:
- **Numerical Features**: Impute missing values with the median.
- **Categorical Features**: Impute missing values with a placeholder string "Missing".


### Numerical -> Median


In [7]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())


### Categorical -> "Missing"


In [8]:
cat_cols = X.select_dtypes(include=['object']).columns
X[cat_cols] = X[cat_cols].fillna("Missing")


## 7. Encode Categorical Features

We use `LabelEncoder` to convert categorical text features into numerical format suitable for most machine learning algorithms.


In [9]:
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))


## 8. Train-Test Split (Stratified)

EDA highlighted a significant class imbalance. To ensure that both the training and evaluation sets are representative of the original data, we use a **stratified split**.


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Train Shape: (472432, 422)
Test Shape: (118108, 422)


## 9. Scaling Numerical Features

Scaling ensures that all numerical features contribute equally to the model, preventing features with larger scales from dominating. We use `StandardScaler` (z-score normalization).


In [15]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

## 10. Save Processed Data

We save the processed datasets to the `data/processed` directory for use in subsequent stages (feature reduction and modeling).


In [16]:
os.makedirs("../data/processed", exist_ok=True)

X_train.to_parquet("../data/processed/X_train.parquet", index=False)
X_test.to_parquet("../data/processed/X_test.parquet", index=False)
pd.DataFrame(y_train).to_parquet("../data/processed/y_train.parquet", index=False)
pd.DataFrame(y_test).to_parquet("../data/processed/y_test.parquet", index=False)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
pd.DataFrame(y_train).to_csv("../data/processed/y_train.csv", index=False)
pd.DataFrame(y_test).to_csv("../data/processed/y_test.csv", index=False)

print("Processed datasets saved!")


Processed datasets saved!


## 11. Save Scaler

We persist the scaler object to ensure consistent scaling when making predictions on new data.


In [17]:
joblib.dump(scaler, "../data/processed/scaler.pkl")
print("Scaler saved!")


Scaler saved!


## 12. Save Feature Metadata

We save the names of numerical and categorical columns to ensure they are handled correctly in the feature reduction and modeling stages.


In [18]:
feature_metadata = {
    'num_cols': num_cols.tolist(),
    'cat_cols': cat_cols.tolist()
}
joblib.dump(feature_metadata, "../data/processed/feature_metadata.pkl")
print("Feature metadata saved!")


Feature metadata saved!


## Final Summary of Preprocessing

- **Merging**: Unified Transaction and Identity data.
- **Cleaning**: Removed features with >90% missing values.
- **Engineering**: Applied log transformation to `TransactionAmt` and extracted decimals.
- **Imputation**: Median for numerical, "Missing" for categorical.
- **Encoding**: Label encoding for categorical variables.
- **Splitting**: Stratified 80/20 split to maintain class balance.
- **Scaling**: Standardized numerical features.

The dataset is now ready for the next stage: **Feature Reduction**.
